<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/06-clasificar-tickets/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 2.6 · Modelos que leen: clasificar tickets

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Cómo un modelo "lee": **contando palabras**.
- Clasificar mensajes de clientes por área.
- Qué pasa cuando aparecen **palabras nuevas** o el cliente mezcla temas.

## 📊 En Excel lo harías así

Atención al Cliente recibe mensajes y debe enviarlos al área correcta. En Excel buscarías palabras:

`=SI(CONTAR.SI(A2,"*factura*")>0, "Facturación", ...)`, con `=SI()` (`IF`) y `=CONTAR.SI()` (`COUNTIF`), y asteriscos como comodines.

Funciona, hasta que alguien escribe *"no me llega el comprobante"*. Hoy dejamos que un modelo aprenda qué palabras indican cada área, y lo comparamos con esa regla.

## 🐍 En Python

### Paso 1: los datos

`tickets.csv` tiene mensajes de clientes y la categoría que les puso una persona. Separamos por fechas, como siempre.

> ⚠️ **Estos mensajes son sintéticos**, armados con unas 60 frases. Tienen mucho menos vocabulario que un negocio real.

In [1]:
import re

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
tickets = pd.read_csv(DATOS + "tickets.csv")

entreno = tickets[tickets["fecha"] < "2025-07-01"]
prueba = tickets[tickets["fecha"] >= "2025-07-01"]

print(len(entreno), len(prueba))
print(entreno["categoria"].value_counts())

tickets[["mensaje", "categoria"]].head()

2633 967
categoria
Consulta comercial    579
Despacho              571
Facturación           511
Producto              489
Cobranza              483
Name: count, dtype: int64


,mensaje,categoria
0,"buenas tardes, ¿desde cuántas cajas hay descue...",Consulta comercial
1,"Joven, anulen esa factura y emitan otra",Despacho
2,"Buenas, el producto vino vencido. Quedo atento.",Producto
3,"Buenos días, enviaron la presentación equivoca...",Producto
4,el aceite llego con la caja rota y manchada,Producto


Si dijeras siempre la categoría más común, acertarías cerca del 22 %. Y una lista de palabras clave, hecha a mano, sería la versión de la fórmula de Excel:

In [3]:
def categoria_por_palabras(mensaje):
    """Una regla hecha a mano, como un =SI() anidado: la primera palabra que coincida decide."""
    reglas = [
        ("Facturación", "factura|boleta|comprobante|ruc|igv|guía"),
        ("Cobranza", "cobr|pag|pague|deud|debo|depósit|deposit|voucher|estado de cuenta"),
        ("Despacho", "pedido|llega|llego|camión|repartidor|despacho|entrega|chofer|mercadería"),
        ("Producto", "vencid|rot|devol|abollad|caja|vino|derram|equivocad"),
    ]
    for categoria, palabras in reglas:
        if re.search(palabras, mensaje.lower()):
            return categoria
    return "Consulta comercial"

In [4]:
por_palabras = prueba["mensaje"].map(categoria_por_palabras)

round((por_palabras == prueba["categoria"]).mean(), 3)

np.float64(0.674)

Cerca del 67 %. No es un límite: con esta lista en otro orden cambia, y otra persona haría otra lista. Pero ya es bastante más que el 22 %, y es la línea base honesta.

### Paso 2: de palabras a números

Un modelo solo entiende números. `CountVectorizer` arma una **tabla enorme**: una columna por cada palabra, una fila por mensaje, y en cada celda cuántas veces aparece. Es parecido a hacer un `=CONTAR.SI()` (`COUNTIF`) por cada palabra.

In [5]:
ejemplo = ["no llegó la factura", "no llegó el pedido", "el pedido llegó roto"]

conteo = CountVectorizer()
tabla = conteo.fit_transform(ejemplo)

pd.DataFrame(tabla.toarray(), columns=conteo.get_feature_names_out(), index=ejemplo)

,el,factura,la,llegó,no,pedido,roto
no llegó la factura,0,1,1,1,1,0,0
no llegó el pedido,1,0,0,1,1,1,0
el pedido llegó roto,1,0,0,1,0,1,1


Fíjate en lo que se pierde: el orden de las palabras. *"No llegó la factura"* y *"la factura no llegó"* quedan iguales. Por eso se le llama *bolsa de palabras*.

### Paso 3: el modelo

Es la misma regresión logística del 1.7, ahora con cinco respuestas posibles. Aprendemos el vocabulario solo en `entreno`, con `.fit_transform()`. En `prueba` usamos `.transform()`: **las palabras nuevas se ignoran, como si no estuvieran**.

In [6]:
conteo = CountVectorizer()
x_entreno = conteo.fit_transform(entreno["mensaje"])
x_prueba = conteo.transform(prueba["mensaje"])

modelo = LogisticRegression(max_iter=2000)
modelo.fit(x_entreno, entreno["categoria"])

print(x_entreno.shape)
print(round(modelo.score(x_prueba, prueba["categoria"]), 3))

(2633, 241)
0.832


Son 241 palabras distintas, y acierta cerca del 83 %, frente al 22 % y al 67 %.

### Paso 4: dónde falla

El promedio esconde lo importante. Separamos los mensajes en grupos. **Podemos hacerlo porque sabemos cómo se escribieron estos mensajes.** Con datos reales, esta separación no se puede hacer tan fácil: habría que leer una muestra de errores.

In [7]:
predicho = pd.Series(modelo.predict(x_prueba), index=prueba.index)
acierta = predicho == prueba["categoria"]

frases_nuevas = (
    "movilidad|la unidad|cpe|emitieron el doc|malogrado|humedad|abono|saldo pendiente|"
    "lista de precios|busco proveedor"
)
nuevas = prueba["mensaje"].str.lower().str.contains(frases_nuevas)
dos_temas = prueba["mensaje"].str.contains("demás,")

print(round(acierta[nuevas & ~dos_temas].mean(), 2), (nuevas & ~dos_temas).sum())
print(round(acierta[dos_temas & ~nuevas].mean(), 2), (dos_temas & ~nuevas).sum())
print(round(acierta[nuevas & dos_temas].mean(), 2), (nuevas & dos_temas).sum())
print(round(acierta[~nuevas & ~dos_temas].mean(), 2), (~nuevas & ~dos_temas).sum())

0.47 135
0.46 72
0.5 44


0.96 716


Son cuatro grupos, y los que tienen ambas cosas son pocos:

- **Palabras nuevas:** cerca del 47 %. Desde julio, algunos clientes escriben con frases que el modelo nunca vio, como *"mi CPE no aparece en SUNAT"*. No sabe qué significan.
- **Dos temas:** cerca del 46 %. Llevan la etiqueta del primer tema, y el conteo de palabras no sabe cuál fue primero.
- **El resto:** cerca del 96 %. **No es un mérito del modelo:** las frases se repiten, y a propósito hay un 4 % mal clasificado, así que ese es casi el techo. Con mensajes reales, espera menos.

Probemos con unos mensajes escritos a mano. Cambia el texto y ejecuta de nuevo:

In [8]:
mios = [
    "no me llegó la factura del pedido",
    "ya hice el abono por Yape",
    "xq me cobraron de más",
    "mi pedido llegó roto y además me cobraron de más",
]

pd.DataFrame({"mensaje": mios, "categoria": modelo.predict(conteo.transform(mios))})

,mensaje,categoria
0,no me llegó la factura del pedido,Facturación
1,ya hice el abono por Yape,Cobranza
2,xq me cobraron de más,Consulta comercial
3,mi pedido llegó roto y además me cobraron de más,Despacho


El segundo acierta por palabras comunes como *"ya"* y *"hice"*: no conoce *"abono"* ni *"Yape"*. El tercero falla, y el cuarto también: no es ni producto ni cobranza. Por eso una persona debe revisar.

### ¿Y ChatGPT?

Hoy existen modelos de lenguaje que entienden mucho mejor el contexto. Entonces, ¿para qué contar palabras? Porque es barato, rápido y **sabes por qué decide**. Además, los datos no salen de tu empresa. Y las preguntas son las mismas con cualquier modelo: ¿contra qué lo comparo?, ¿dónde falla? y ¿qué pasa cuando cambian las palabras?

## 🤖 Pídeselo a la IA

> *"Tengo una lista de 200 mensajes de clientes inventados. Quiero separarlos por área. ¿Cómo evalúo si un modelo lo hace bien? Dame una lista de chequeo sencilla."*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| `=CONTAR.SI()` (`COUNTIF`) por cada palabra | `CountVectorizer().fit_transform(textos)` |
| Usar las columnas de entrenamiento | `conteo.transform(textos)` |
| Clasificar el mensaje | `modelo.predict(...)` |
| Ver aciertos por grupo | `acierta[filtro].mean()` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/06-clasificar-tickets/ejercicio.ipynb) y prueba otros mensajes. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/06-clasificar-tickets/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0